# 01 — Anatomy of the target: when TAXITIME is not a taxi time

This notebook produces the paper's figures 1-3 from the pipeline caches
(`data/dep_features_cache_v15.parquet`, `data/val_preds_v15.parquet`).
Prerequisite: run `legacy/reproduce.py` once (caches are created on the first run).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("..") / "data"
dep = pd.read_parquet(DATA / "dep_features_cache_v15.parquet")
print(f"{len(dep):,} departures, {dep.shape[1]} columns")

## Figure 1 — Two off-block definitions: airport vs Network Manager bias

`delta_mvt_aobt3_s` is the taxi time as seen by NM (`MVT − AOBT_3`);
`TAXITIME` is the airport's. Their median gap per airport shows the two
systems do not measure the same event.

In [ ]:
d = dep[(dep["is_corrupt"] == 0) & dep["delta_mvt_aobt3_s"].notna()].copy()
d["gap"] = d["delta_mvt_aobt3_s"] - d["TAXITIME_SEC_mvt"]
stats = d.groupby("ADEP_mvt", observed=True)["gap"].agg(["median", "std", "size"]).sort_values("median")
print(stats.round(0))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(stats.index.astype(str), stats["median"], yerr=stats["std"] / np.sqrt(stats["size"]),
       color="#3B82F6")
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Median gap, NM − airport (s)")
ax.set_title("Bias between NM off-block (AOBT_3) and airport off-block, clean rows")
plt.tight_layout(); plt.show()

## Figure 2 — The silent schedule substitution

4.8 % of departures have `BLOCK = SCHED` within 10 s; their `TAXITIME` then
equals `MVT − SCHED` (delay + real taxi), not a taxi time.

In [ ]:
rate = dep.groupby("ADEP_mvt", observed=True)["is_corrupt"].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(rate.index.astype(str), 100 * rate.values, color="#10B981")
ax.axhline(100 * dep["is_corrupt"].mean(), color="black", ls="--", lw=0.8,
           label=f"mean {100 * dep['is_corrupt'].mean():.1f} %")
ax.set_ylabel("Share of departures with BLOCK = SCHED (%)")
ax.set_title("Off-block substitution rate by airport, 2025")
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Check the identity TAXITIME = MVT − SCHED on substituted rows
c = dep[dep["is_corrupt"] == 1]
gap = (c["TAXITIME_SEC_mvt"] - c["delta_mvt_sched_s"]).abs()
print(f"|TAXITIME − (MVT − SCHED)| on substituted rows: "
      f"median {gap.median():.0f} s, p95 {gap.quantile(0.95):.0f} s")

## Figure 3 — A handful of rows decides the RMSE

Validation MSE decomposition (January + July 2025, final pipeline
predictions) by regime, and the concentration curve.

In [ ]:
v = pd.read_parquet(DATA / "val_preds_v15.parquet")
v["se"] = (v["pred"] - v["y"]) ** 2
N = len(v); mse = v["se"].sum() / N
v["regime"] = np.select(
    [v["is_corrupt"] == 1, (v["y"] - 86400).between(30, 7200), v["y"] > 3600],
    ["block = sched", "+1 day bug", "tail > 1 h"], "normal")
share = v.groupby("regime")["se"].sum() / v["se"].sum()
n_per = v.groupby("regime").size()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
a1.barh(share.index, 100 * share.values, color="#3B82F6")
for i, (p, n) in enumerate(zip(share.values, n_per.values)):
    a1.text(100 * p + 1, i, f"{n:,} rows", va="center", fontsize=9)
a1.set_xlabel("Share of MSE (%)"); a1.set_title(f"MSE by regime (RMSE {mse**0.5:.0f} s)")

s = v["se"].sort_values(ascending=False).cumsum() / v["se"].sum()
ks = np.arange(1, 2001)
a2.plot(ks, 100 * s.iloc[:2000].values, color="#10B981")
a2.set_xscale("log"); a2.set_xlabel("k worst rows (log)")
a2.set_ylabel("Cumulative share of MSE (%)")
a2.set_title("Concentration: 10 rows ≈ 30 % of the MSE")
a2.axvline(10, color="black", ls=":", lw=0.8)
plt.tight_layout(); plt.show()

Reading: more than half of the MSE comes from the normal regime (the data
floor), a quarter from a few unsigned tail rows, and the substituted regimes
are largely handled by the mixture.